# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** A/B Testing, Optimization & Statistical Inference

---
*Nota Institucional: Este notebook simula la conclusión empírica de un experimento A/B (Prueba de Hipótesis) sobre el proceso de Onboarding de nuestro Hardware. Queremos determinar, con rigor matemático, si una variante específica del flujo inicial de la aplicación móvil tiene un impacto causal, estadísticamente significativo, en la Tasa de Conversión a Suscripciones de pago (Hardware-to-Subscription Attach Rate).*

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette("Blues")

### 1. Ingesta de Datos del Experimento (Asignación Aleatoria)
Evaluamos la partición aleatoria entre el Flujo de Control (Onboarding Clásico) y la Variante (Flujo de Onboarding Rediseñado).

In [ ]:
df_users = pd.read_csv('../data/hardware_users.csv')
df_subs = pd.read_csv('../data/subscriptions.csv')

df_subs['Has_Subscribed'] = 1
df_ab = pd.merge(df_users, df_subs[['UserID', 'Has_Subscribed']], on='UserID', how='left')
df_ab['Has_Subscribed'] = df_ab['Has_Subscribed'].fillna(0).astype(int)

print("[*] Resumen de Muestra del Experimento A/B:\n")
print(df_ab['OnboardingVersion'].value_counts())
df_ab.head()

### 2. Resultados Empíricos (Sample Conversion Rates)
Agrupamos los resultados binarios para observar la diferencia absoluta superficial antes de aplicar los umbrales de sensibilidad del Teorema Central del Límite.

In [ ]:
ab_summary = df_ab.groupby('OnboardingVersion').agg(
    Exposiciones=('UserID', 'count'),
    Conversiones=('Has_Subscribed', 'sum'),
    Conversion_Rate=('Has_Subscribed', 'mean')
)

ab_summary['Conversion_Rate_Pct'] = ab_summary['Conversion_Rate'] * 100
ab_summary

### 3. Prueba de Hipótesis: Z-Test de Dos Proporciones
Establecemos:
- Hipótesis Nula $H_0$: $p_{variante} = p_{control}$ (El nuevo onboarding no cambia nada).
- Hipótesis Alternativa $H_1$: $p_{variante} > p_{control}$ (El nuevo onboarding mejora la conversión).
- Nivel de Significancia (Alfa $\alpha$): $0.05$ (Confiabilidad del 95%).

In [ ]:
control_results = df_ab[df_ab['OnboardingVersion'] == 'Control']['Has_Subscribed']
variant_results = df_ab[df_ab['OnboardingVersion'] == 'Variant']['Has_Subscribed']

n_con = control_results.count()
n_var = variant_results.count()
successes = [control_results.sum(), variant_results.sum()]
nobs = [n_con, n_var]

z_stat, pval = proportions_ztest(successes, nobs=nobs)
(lower_con, lower_var), (upper_con, upper_var) = proportion_confint(successes, nobs=nobs, alpha=0.05)

print(f"[*] Z-Statistic (Estadístico de prueba): {z_stat:.4f}")
print(f"[*] P-Value (Valor p): {pval:.4e}")
print("-------------------------------------------")
print(f"[-] Control 95% Intervalo de Confianza: [{lower_con:.4f}, {upper_con:.4f}]")
print(f"[+] Variant 95% Intervalo de Confianza: [{lower_var:.4f}, {upper_var:.4f}]")

### 4. Interpretación y Visualización del Rango de Incertidumbre
La regla de decisión de la estadística frecuentista dicta que si $p < \alpha$, rechazamos enérgicamente $H_0$.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

rates = [control_results.mean(), variant_results.mean()]
errors = [ (upper_con-lower_con)/2, (upper_var-lower_var)/2 ]

bars = ax.bar(['Grupo Control', 'Grupo Variante'], rates, yerr=errors, capsize=10, color=['#78909c', '#29b6f6'], alpha=0.8)
ax.set_title("Efectividad de Conversión A/B (Intervalos de Confianza al 95%)", fontsize=14, pad=15)
ax.set_ylabel("Tasa de Conversión Promedio")

# Añadir etiquetas
for index, bar in enumerate(bars):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2,
            f"{rates[index]*100:.2f}%", ha='center', color='white', fontweight='bold', fontsize=14)

if pval < 0.05:
    plt.suptitle("\nResultado de Negocio: Rechazamos la Hipótesis Nula (Mejora Significativa)", color='lime')
else:
    plt.suptitle("\nResultado de Negocio: Fallamos al rechazar H0 (Inconclusivo)", color='yellow')

plt.show()